## ragas评估

### rag pipline

In [1]:
import warnings
warnings.filterwarnings('ignore')
from model import RagEmbedding, RagLLM, QwenLLM
from langchain_chroma import Chroma
import chromadb

In [2]:
llm = RagLLM()

In [3]:
prompt_template = """
你是企业员工助手，熟悉公司考勤和报销标准等规章制度，需要根据提供的上下文信息context来回答员工的提问。\
请直接回答问题，如果上下文信息context没有和问题相关的信息，请直接回答[不知道,请咨询HR] \
问题：{question} 
"{context}"
回答：
"""

In [9]:
chroma_client = chromadb.HttpClient(host="localhost", port=8000)

In [5]:
import os
from langchain_community.embeddings import DashScopeEmbeddings

os.environ["DASHSCOPE_API_KEY"] = "sk-ws-H.PHPPRYP.LbX3.MEYCIQDIz-wrCcL4dQUTaV-L3TSXK2Dld6oqLJn5SO8QKWfRKAIhANQIGIFNNXJZnRqUUkYNEonmfJRLZvhkg17LY8cvkLeu"

embedding = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

In [6]:
# embedding_model = RagEmbedding()
from langchain_community.embeddings import DashScopeEmbeddings

embedding = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

In [10]:
zhidu_db = Chroma("zhidu_db_bailian",
                  embedding,
                  client=chroma_client)

In [11]:
def run_rag_pipline_without_stream(query, k=3):
    related_docs = zhidu_db.similarity_search(query, k=k)
    context_list = [f"上下文{i+1}: {doc.page_content} \n" \
                         for i, doc in enumerate(related_docs)]
    context = "\n".join(context_list)

    llm_prompt = prompt_template.replace("{question}", query).replace("{context}", context)
    response = llm(llm_prompt, stream=False)
    return response, context_list

### 构建评估数据集
- 问题
- 标准答案
- 上下文信息
- 生成的答案

In [12]:
questions = [
    "伙食补助费标准是什么?",
    "出差可以买意外保险吗？需要自己购买吗",
]
ground_truths = [
    "伙食补助费标准: 西藏、青海、新疆 120元/人、天 其他省份 100元/人、天",
    "出差可以购买交通意外保险，由单位统一购买，不再重复购买",
]

In [13]:
answers = []
contexts = []

for query in questions:
    response, context_list = run_rag_pipline_without_stream(query, k=3)
    answers.append(response)
    contexts.append(context_list)


In [14]:
from datasets import Dataset

In [15]:
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

In [16]:
dataset = Dataset.from_dict(data)

In [17]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

In [18]:
from ragas import evaluate
from ragas import RunConfig

In [25]:
# eval_llm = QwenLLM()
# embedding_model = RagEmbedding()
# eval_embedding_fn = embedding_model.get_embedding_fun()
eval_llm = QwenLLM()
eval_embedding_fn = embedding

In [26]:
config = RunConfig(timeout=1200, log_tenacity=True)

In [28]:
import importlib
import model

print("加载的文件：", model.__file__)
importlib.reload(model)

# 必须重新创建对象，明确使用重新加载后的类
eval_llm = model.QwenLLM()
eval_embedding_fn = embedding

print("评估模型类型：", eval_llm._llm_type)
print("测试结果：", eval_llm.invoke("请只回复：测试成功"))

加载的文件： /Users/dengyixuan/learnLargeModel/Projects/pythonProject/rag_full_stack_course_notebooks/notebook/model.py
评估模型类型： deepseek_rag_evaluator
测试结果： 测试成功


In [29]:
result = evaluate(
    dataset = dataset,
    llm=eval_llm,
    embeddings=eval_embedding_fn,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    raise_exceptions=True,
    run_config=config
)

df = result.to_pandas()

Evaluating: 100%|██████████| 8/8 [00:03<00:00,  2.15it/s]


In [30]:
df

,question,answer,contexts,ground_truth,context_precision,context_recall,faithfulness,answer_relevancy
0,伙食补助费标准是什么?,伙食补助费标准如下：\n- 西藏、青海、新疆：120元/人、天\n- 其他省份：100元/人...,[上下文1: 2\n<table><caption>伙食补助费参考以下标准：</captio...,伙食补助费标准: 西藏、青海、新疆 120元/人、天 其他省份 100元/人、天,1.0,1.0,1.0,0.954386
1,出差可以买意外保险吗？需要自己购买吗,可以购买。根据规定，乘坐飞机、火车、轮船等交通工具的，每人次可以购买交通意外保险一份；如果由...,[上下文1: 差旅费用标准\n差旅费开支范围包括工作人员临时到常驻地以外地区公务出差所发生的...,出差可以购买交通意外保险，由单位统一购买，不再重复购买,1.0,1.0,1.0,0.766985
